# (04) additive decoder play

project = ```iP-VAE```, host = ```yoru```, device = ```cuda:0```

**Motivation**: <br>

Use their code to fit on gray balls dataset.

In [1]:
# HIDE CODE


import os, sys
from IPython.display import display

# tmp & extras dir
git_dir = os.path.join(os.environ['HOME'], 'Dropbox/git')
extras_dir = os.path.join(git_dir, 'jb-progress-2025/_extras')
fig_base_dir = os.path.join(git_dir, 'jb-progress-2025/figs')
tmp_dir = os.path.join(git_dir, 'jb-progress-2025/tmp')

# GitHub
sys.path.insert(0, os.path.join(git_dir, '_IterativeVAE'))
from figures.analysis import plot_convergence
from figures.imgs import plot_weights
from figures.fighelper import *
from main.train import *

# warnings, tqdm, & style
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=UserWarning)
from rich.jupyter import print
%matplotlib inline
set_style()

In [2]:
device_idx = 0
device = f'cuda:{device_idx}'

print(f"device: {device}  ———  host: {os.uname().nodename}")

device: cuda:0  ———  host: yoru

## Import their stuff

In [3]:
sys.path.insert(0, os.path.join(git_dir, '[git-cloned]/additive_decoder_extrapolation'))

In [4]:
from algorithms.base_auto_encoder import AE
from algorithms.additive_auto_encoder import AE_Additive
from data.balls_dataset_loader import sample_base_data_loaders

In [5]:
defaults = dict(
    method_type='ae_additive',
    data_dim=200,
    latent_dim=2,
    total_blocks=2,
    latent_case='balls_supp_l_shape',
    train_size=10_000,
    batch_size=64,
    lr=5e-4,
    weight_decay=5e-4,
    num_epochs=1000,
    seed=0,
    input_normalization='none',
)
kwargs = {'num_workers': 0, 'pin_memory': False} 

args = defaults.copy()

In [7]:
# train_dataset, val_dataset, test_dataset= sample_base_data_loaders(
#     latent_case= args['latent_case'], 
#     num_balls= args['total_blocks'],
#     train_size= args['train_size'],
#     batch_size= args['batch_size'], 
#     input_normalization= args['input_normalization'],
#     kwargs=kwargs,
# )

NameError: name 'sample_base_data_loaders' is not defined

In [ ]:
method= AE_Additive(args, train_dataset, val_dataset, test_dataset, seed=seed, device= device)

In [ ]:
#Common imports
import sys
import os
import argparse
import random
import copy

import torch
from torch import nn, optim
from torch.nn import functional as F
from torchvision import datasets, transforms
from torchvision.utils import save_image
from torch.autograd import Variable

import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.decomposition import FastICA

#Algorithms
from algorithms.base_auto_encoder import AE
from algorithms.additive_auto_encoder import AE_Additive

#DataLoaders
from data.balls_dataset_loader import BallsDataLoader
from data.balls_dataset_loader import sample_base_data_loaders

# Input Parsing




#GPU
if cuda_device == -1:
    device= torch.device("cpu")
else:
    device= torch.device("cuda:" + str(cuda_device))
    
if device:
    kwargs = {'num_workers': 0, 'pin_memory': False} 
else:
    kwargs= {}

#Seed values
random.seed(seed*10)
np.random.seed(seed*10) 
torch.manual_seed(seed*10)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed*10)
        
    
# Load Dataset
train_dataset, val_dataset, test_dataset= sample_base_data_loaders(
                                                                  latent_case= latent_case, 
                                                                  num_balls= total_blocks,
                                                                  train_size= train_size,
                                                                  batch_size= batch_size, 
                                                                  input_normalization= input_normalization,
                                                                  kwargs=kwargs
                                                                 )

#Load Algorithm
if method_type == 'ae_base':
    method= AE(args, train_dataset, val_dataset, test_dataset, seed=seed, device= device)    
elif method_type == 'ae_additive':
    method= AE_Additive(args, train_dataset, val_dataset, test_dataset, seed=seed, device= device)    
else:
    print('Error: Incorrect method type')
    sys.exit(-1)

# Training
method.train()